> **このノートはColabでは動きません。**
> FreeCAD / Blender 本体（デスクトップアプリ）が必要です。GitHub上では閲覧のみ。コードは各アプリのPythonコンソール／Scriptingに貼り付けて使います。
>
> （Colabで動くのは統計・マーケ系と `SolidPython` のノートです）

# ホビー-01. FreeCAD で陶芸の器をつくる

**FreeCAD** は無料のCADソフト。正確な寸法（mm単位）で設計でき、
ろくろで作る器のような「回転体」がとても得意です。

このノートは **設計図と考え方** を説明します。
コードは FreeCAD の中の Python コンソールに貼り付けて動かします。

> このノート自体は実行しなくてOK。FreeCADを開いて、コードをコピーして使います。

> **このツールはこんな時に**：寸法が命のもの。**蓋がぴったり合う**部品、ろくろ用の**こて/テンプレ**、粘土の**抜き型**、石こう型の位置決め。→ 道具づくりは [04 実用ツール](04_%E5%AE%9F%E7%94%A8%E3%83%84%E3%83%BC%E3%83%AB%E3%82%92%E3%81%A4%E3%81%8F%E3%82%8B.ipynb) も参照。

## 0. 準備

1. [freecad.org](https://www.freecad.org/) から FreeCAD をインストール
2. メニュー **表示 → パネル → Python コンソール** にチェック
3. 下に出てくる黒い入力欄に、これから出てくるコードを貼り付けます

## 1. 器づくりの考え方：回転体

茶碗や湯のみは、**断面（半分の輪郭）を1本の軸のまわりに360°回す**とできます。
ろくろと同じ発想です。

```
  輪郭(プロファイル)        回転        器
   |‾\                                  ___
   |  \         ──360°回す──>          /   \
   |__/                                \___/
```

やることは「断面を点でかく → 面にする → 回す」の3ステップだけです。

### 陶芸ならではの注意：収縮率

粘土は乾燥・焼成で **10〜15%ちぢみます**。焼き上がりで口径120mmにしたいなら、
モデルは `120 ÷ (1 - 0.13) ≈ 138mm` で作ります。これは比の良い練習にもなります。

```python
finish = 120          # 焼き上がりで欲しい口径(mm)
shrink = 0.13         # 収縮率13%
model = finish / (1 - shrink)
print(round(model, 1), 'mm で設計する')
```

## 2. 茶碗をつくるスクリプト

FreeCAD の Python コンソールに貼り付けてください。
単位は mm。点は `(半径, 高さ)` の順で、内側と外側をぐるりと一周する閉じた輪郭です。

In [ ]:
# === FreeCADのPythonコンソールに貼り付け ===
import FreeCAD as App        # FreeCAD本体
import Part                  # 立体（パート）を作るモジュール
from FreeCAD import Vector   # 3D座標 (x, y, z)

doc = App.newDocument('Chawan')   # 新しいドキュメントを作る

# 断面プロファイル (半径, 高さ) ※閉じた輪郭にする
pts = [
    Vector(0,   0, 0),   # 底の中心（外）
    Vector(45,  0, 0),   # 底の外ふち
    Vector(62, 70, 0),   # 口の外ふち（口が広がる）
    Vector(56, 70, 0),   # 口の内ふち → 肉厚6mm
    Vector(39,  8, 0),   # 見込み（内側の壁）
    Vector(0,   8, 0),   # 内側の底
    Vector(0,   0, 0),   # 閉じる
]

wire = Part.makePolygon(pts)   # 点をつないだ輪郭線（ワイヤ）
face = Part.Face(wire)         # 輪郭を面にする
# Y軸(高さ方向)のまわりに360°回転
bowl = face.revolve(Vector(0,0,0), Vector(0,1,0), 360)   # 面を回して立体（器）に

Part.show(bowl, 'Chawan')      # 画面に表示
doc.recompute()                # 再計算して反映
# 全体を見る: メニュー 表示→標準ビュー→軸測 / マウスホイールでズーム

実行すると、画面に茶碗が現れます！ 数字を変えると形が変わります。

- `62 → 50`：口がすぼまった湯のみ風に
- 高さ `70 → 110`：背の高いカップに
- 肉厚（外と内の半径の差）を大きく：どっしりした器に

## 3. 高台（こうだい）をつける

器の底の輪っか状の脚を「高台」といいます。
小さな円柱を引き算（くり抜き）して表現できます。

In [ ]:
# 底をへこませて高台をつくる（コンソールに追記）
cut = Part.makeCylinder(33, 6, Vector(0,0,0), Vector(0,1,0))   # くり抜く円柱（半径33・高さ6）
chawan_with_foot = bowl.cut(cut)             # 器から円柱を引き算
Part.show(chawan_with_foot, 'Chawan_kodai')  # 表示
doc.recompute()                              # 再計算

## 4. STLで書き出して3Dプリント／型に使う

メニュー **ファイル → エクスポート** で `.stl` を選ぶか、コードでも出せます。

In [ ]:
# 書き出し（パスは自分の環境に合わせて）
bowl.exportStl('/Users/あなた/Desktop/chawan.stl')   # STL形式で保存（3Dプリンタ用）
print('保存しました')

---
## チャレンジ課題

**課題1.** 焼き上がりで口径100mm・高さ60mmの湯のみにしたい。
収縮率12%を考えて、モデルの寸法を計算し、`pts` を書き換えよう。

**課題2.** 口が外に大きく開く「抹茶碗」と、まっすぐな「そば猪口」を作り分けよう。

**課題3.**（発展）`pts` の中間に点を増やして、ふくらみのある曲線的な器にしよう。

> 次は同じ器を **Blender** で、より自由な曲面として作ってみます（02のノートへ）。

## 出力ファイルの見方・操作

**可視化は FreeCAD の中で完結**します（上で `Part.show()` した器を、その場で回せます）。
- 回転：マウス中ボタンドラッグ／画面右上のナビゲーションキューブ
- ズーム：マウスホイール
- 書き出し：メニュー **ファイル → エクスポート** で `.stl` を選ぶ（コードの `exportStl` でも可）

### `.stl` ファイルの見方・使い道（共通）

`.stl` は3D形状の標準フォーマット。次のように扱えます。
- **見るだけ**：ダブルクリックでOSの3Dビューア（Mac=プレビュー / Win=3Dビューア）。または無料オンラインビューア（`viewstl.com` などにファイルをドラッグ）。
- **編集**：Blender や FreeCAD に読み込んで形を調整。
- **3Dプリント**：スライサー（**Cura** / **PrusaSlicer**、無料）で `.stl` を開く → 印刷設定 → プリンタへ。

> **陶芸での使い道**：プリントした器そのものは焼けませんが、**石こう型の原型**や、ろくろ・たたら作りの**テンプレート/ゲージ**として活用できます。

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""      # ← あなたの名前（例: 山田）
LOG_URL = ""   # ← 記録用URL（配布時に設定。空なら記録せず表示のみ）
NOTEBOOK = "07_ホビー_陶芸3D/01_FreeCADで器をつくる"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")